# MaisonDeLUX data pipeline

This is the single orchestration notebook. Production logic lives in `ml/src/`; the notebook never invents missing values and never substitutes collection time for publication time. Commercial listing adapters remain disabled unless their recorded policy gate permits collection.

In [ ]:
from pathlib import Path
import json, subprocess, sys
import pandas as pd

ROOT = Path.cwd().resolve()
while ROOT.name != 'MaisonDeLUX' and ROOT.parent != ROOT:
    ROOT = ROOT.parent
assert (ROOT / 'ml' / 'src' / 'pipeline.py').exists(), 'Run this notebook from inside the repository.'
RUN_GEOGRAPHY_REFRESH = False   # downloads attributed open geographic sources
RUN_POLICY_PILOTS = False       # bounded robots/API preflights; never bypasses restrictions
RUN_RECOVERY_PIPELINE = False   # safe to rerun; canonical outputs replace prior generated outputs
MODEL_MIN_LISTINGS = 30

## 1. Load configuration and optional legal-source pilots

In [ ]:
def run_module(module):
    completed = subprocess.run([sys.executable, '-m', module], cwd=ROOT, check=True, text=True, capture_output=True)
    print(completed.stdout.strip())

if RUN_GEOGRAPHY_REFRESH:
    run_module('ml.src.geography.build_reference')
if RUN_POLICY_PILOTS:
    run_module('ml.src.scraping.pilot')
json.loads((ROOT / 'reports' / 'scraping' / 'source_policy_audit.json').read_text(encoding='utf-8'))

## 2. Recover, normalize, validate, deduplicate, enrich and export

In [ ]:
state_path = ROOT / 'data' / 'interim' / 'pipeline_state.json'
if RUN_RECOVERY_PIPELINE:
    run_module('ml.src.pipeline')
state = json.loads(state_path.read_text(encoding='utf-8'))
state  # completed stages are checkpointed; reruns recover from archived inputs without duplicate requests

## 3. Concise quality summaries

In [ ]:
raw = pd.read_parquet(ROOT / 'data' / 'raw' / 'maisondelux_raw.parquet')
clean = pd.read_parquet(ROOT / 'data' / 'processed' / 'maisondelux_clean.parquet')
rejected = pd.read_csv(ROOT / 'data' / 'processed' / 'maisondelux_rejected.csv', low_memory=False)
pd.DataFrame({
    'metric': ['raw rows', 'valid unique rows', 'rejected/warning rows', 'sources', 'cities', 'neighborhoods'],
    'value': [len(raw), len(clean), len(rejected), raw.source.nunique(), raw.city.nunique(), raw.neighborhood.nunique()]
})

In [ ]:
(clean.groupby(['region', 'city'], dropna=False).size().rename('valid_listings')
 .sort_values(ascending=False).head(25).to_frame())

In [ ]:
pd.Series('|'.join(rejected.validation_reasons.fillna('')).split('|')).value_counts().head(20).rename('rows')

## 4. Verify final exports

Excel files are generated once after collection (never during scraping) and contain `all_rows`, `valid_rows`, `rejected_rows`, source/city/quality summaries, and scraping errors.

In [ ]:
expected = [
    'data/raw/maisondelux_raw.csv', 'data/raw/maisondelux_raw.xlsx', 'data/raw/maisondelux_raw.parquet',
    'data/processed/maisondelux_clean.csv', 'data/processed/maisondelux_clean.xlsx', 'data/processed/maisondelux_clean.parquet',
    'data/processed/maisondelux_rejected.csv', 'data/geographic/morocco_regions.geojson',
    'data/geographic/morocco_cities.geojson', 'data/geographic/morocco_neighborhoods.geojson',
    'reports/data_quality/data_quality_report.md', 'reports/scraping/source_coverage_report.md',
    'reports/scraping/geographic_coverage_report.md', 'reports/scraping/historical_coverage_report.md'
]
verification = pd.DataFrame([{'path': item, 'exists': (ROOT/item).exists(), 'bytes': (ROOT/item).stat().st_size if (ROOT/item).exists() else 0} for item in expected])
assert verification.exists.all(), verification.loc[~verification.exists]
verification

## Modeling boundary

Train only on `maisondelux_clean.parquet`. Exclude `price_per_m2` when predicting `price_mad`; it directly contains the target. Split by time only where a true publication date exists, otherwise use grouped spatial/source-aware validation.